# AeroVision LK — Baseline YOLOv8s Training

**Week 1 | Day 5–7**  
**Model:** YOLOv8s @ 640px input  
**Dataset:** VisDrone2019-DET (9-class remapped)  
**Goal:** Establish the pre-SAHI benchmark.

---
**Context from EDA:**  
78.7% of objects in VisDrone are < 50×50 px at native resolution.  
After YOLO's 640px letterbox resize, these shrink to ~33px or less.  
This notebook measures exactly how much that hurts detection performance.

**Expected results:**  
- mAP@0.5 ≈ 28%  
- mAP@0.5:0.95 ≈ 16%  

This is the baseline to beat with SAHI in `03_sahi_inference.ipynb`.

---

In [1]:
from pathlib import Path
import shutil
import yaml
import json
import datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
import mlflow
from ultralytics import YOLO

EXPERIMENT_NAME = "aerovision-baseline"

# ── Project root ─────────────────────────────────────────────────────────────
def find_project_root():
    for candidate in [Path.cwd(), Path.cwd().parent]:
        if (candidate / 'data' / 'VisDrone_Dataset').exists():
            return candidate
    raise RuntimeError('Cannot find project root. Run from 01_aerovision_lk/ or research/')

PROJECT_ROOT = find_project_root()
DATASET_YAML = PROJECT_ROOT / 'data' / 'VisDrone_Dataset' / 'visdrone.yaml'
WEIGHTS_DIR  = PROJECT_ROOT / 'weights'
RESULTS_DIR  = PROJECT_ROOT / 'runs' / 'baseline'
ANALYSIS_DIR = PROJECT_ROOT / 'analysis'
FIGURES_DIR  = PROJECT_ROOT / 'reports' / 'figures'

for d in [WEIGHTS_DIR, RESULTS_DIR, ANALYSIS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ['pedestrian', 'people', 'bicycle', 'car', 'van',
               'truck', 'three_wheeler', 'bus', 'motor']
NC = len(CLASS_NAMES)

COLORS_RGB = [
    (255,  59,  59),   # 0 pedestrian
    (255, 149,   0),   # 1 people
    (255, 214,   0),   # 2 bicycle
    ( 52, 199,  89),   # 3 car
    (  0, 190, 235),   # 4 van
    (  0, 122, 255),   # 5 truck
    (175,  82, 222),   # 6 three_wheeler
    (255,  45, 143),   # 7 bus
    (162, 132,  94),   # 8 motor
]
COLORS_BGR = [(b, g, r) for r, g, b in COLORS_RGB]

print(f'Project root : {PROJECT_ROOT}')
print(f'Dataset yaml : {DATASET_YAML}')
print(f'Weights dir  : {WEIGHTS_DIR}')
print(f'Results dir  : {RESULTS_DIR}')

c:\Users\Yasindu\.conda\envs\aerovision\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root : c:\Users\Yasindu\Desktop\Stuff\5.ML Projects\deep-learning-projects\01_aerovision_lk
Dataset yaml : c:\Users\Yasindu\Desktop\Stuff\5.ML Projects\deep-learning-projects\01_aerovision_lk\data\VisDrone_Dataset\visdrone.yaml
Weights dir  : c:\Users\Yasindu\Desktop\Stuff\5.ML Projects\deep-learning-projects\01_aerovision_lk\weights
Results dir  : c:\Users\Yasindu\Desktop\Stuff\5.ML Projects\deep-learning-projects\01_aerovision_lk\runs\baseline


---
## 1. Dataset Verification

In [2]:
# ── Verify dataset yaml exists ────────────────────────────────────────────────
assert DATASET_YAML.exists(), f'visdrone.yaml not found at {DATASET_YAML}'

with open(DATASET_YAML) as f:
    ds_cfg = yaml.safe_load(f)

print('visdrone.yaml contents:')
print(yaml.dump(ds_cfg, default_flow_style=False))

# ── Count images per split ────────────────────────────────────────────────────
dataset_root = DATASET_YAML.parent
splits = {
    'train': dataset_root / ds_cfg.get('train', 'VisDrone2019-DET-train/images'),
    'val':   dataset_root / ds_cfg.get('val',   'VisDrone2019-DET-val/images'),
}

print('Split verification:')
print(f'  {"Split":<8} {"Images":>8}  Path')
print('  ' + '-' * 60)
for split, img_dir in splits.items():
    count = len(list(img_dir.glob('*.jpg'))) if img_dir.exists() else 0
    status = 'OK' if count > 0 else 'MISSING'
    print(f'  {split:<8} {count:>8,}  {img_dir}  [{status}]')
    assert count > 0, f'No images found in {img_dir}'

print()
print(f'Classes ({ds_cfg["nc"]}): {list(ds_cfg["names"].values()) if isinstance(ds_cfg["names"], dict) else ds_cfg["names"]}')
print()
print('Dataset verification passed. Ready to train.')

visdrone.yaml contents:
names:
  0: pedestrian
  1: people
  2: bicycle
  3: car
  4: van
  5: truck
  6: three_wheeler
  7: bus
  8: motor
nc: 9
path: data/VisDrone_Dataset
test: VisDrone2019-DET-test-dev/images
train: VisDrone2019-DET-train/images
val: VisDrone2019-DET-val/images

Split verification:
  Split      Images  Path
  ------------------------------------------------------------
  train       6,471  c:\Users\Yasindu\Desktop\Stuff\5.ML Projects\deep-learning-projects\01_aerovision_lk\data\VisDrone_Dataset\VisDrone2019-DET-train\images  [OK]
  val           548  c:\Users\Yasindu\Desktop\Stuff\5.ML Projects\deep-learning-projects\01_aerovision_lk\data\VisDrone_Dataset\VisDrone2019-DET-val\images  [OK]

Classes (9): ['pedestrian', 'people', 'bicycle', 'car', 'van', 'truck', 'three_wheeler', 'bus', 'motor']

Dataset verification passed. Ready to train.


---
## 2. MLflow Experiment Setup

In [3]:
mlflow.set_tracking_uri((PROJECT_ROOT / 'mlruns').as_uri())
mlflow.set_experiment('aerovision-lk')

active_exp = mlflow.get_experiment_by_name('aerovision-lk')
print(f'MLflow tracking URI : {mlflow.get_tracking_uri()}')
print(f'Experiment name     : aerovision-lk')
print(f'Experiment ID       : {active_exp.experiment_id if active_exp else "(will be created on first run)"}')
print()
print('Run: mlflow ui --backend-store-uri mlruns  (from project root) to view results.')

MLflow tracking URI : file:///c:/Users/Yasindu/Desktop/Stuff/5.ML%20Projects/deep-learning-projects/01_aerovision_lk/mlruns
Experiment name     : aerovision-lk
Experiment ID       : 333239585452034053

Run: mlflow ui --backend-store-uri mlruns  (from project root) to view results.


c:\Users\Yasindu\.conda\envs\aerovision\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


---
## 3. Baseline Training

**Config:** YOLOv8s · 50 epochs · 640px · batch 8 · GPU 0  
Training takes ~3–5 hours on a single GPU depending on hardware.  
Results are saved to `runs/baseline/yolov8s_640/`.

In [4]:
import yaml as _yaml

# ── Resolve visdrone.yaml to absolute paths ───────────────────────────────────
# Ultralytics resolves relative 'path' from its own DATASETS_DIR, not the
# project root. Write a resolved copy with an absolute path before training.
with open(DATASET_YAML) as f:
    _cfg = _yaml.safe_load(f)

_cfg['path'] = str(PROJECT_ROOT / 'data' / 'VisDrone_Dataset')
DATASET_YAML_ABS = PROJECT_ROOT / 'data' / 'VisDrone_Dataset' / 'visdrone_abs.yaml'
with open(DATASET_YAML_ABS, 'w') as f:
    _yaml.dump(_cfg, f, default_flow_style=False, sort_keys=False)

print(f'Resolved yaml → {DATASET_YAML_ABS}')
print(f'  path : {_cfg["path"]}')
print()

# ── Training params ───────────────────────────────────────────────────────────
TRAIN_PARAMS = {
    'data':     str(DATASET_YAML_ABS),
    'epochs':   50,
    'imgsz':    640,
    'batch':    8,
    'device':   0,
    'project':  str(RESULTS_DIR),
    'name':     'yolov8s_640',
    'exist_ok': True,
    'verbose':  True,
}

print('Training parameters:')
for k, v in TRAIN_PARAMS.items():
    print(f'  {k:<12}: {v}')

Resolved yaml → c:\Users\Yasindu\Desktop\Stuff\5.ML Projects\deep-learning-projects\01_aerovision_lk\data\VisDrone_Dataset\visdrone_abs.yaml
  path : c:\Users\Yasindu\Desktop\Stuff\5.ML Projects\deep-learning-projects\01_aerovision_lk\data\VisDrone_Dataset

Training parameters:
  data        : c:\Users\Yasindu\Desktop\Stuff\5.ML Projects\deep-learning-projects\01_aerovision_lk\data\VisDrone_Dataset\visdrone_abs.yaml
  epochs      : 50
  imgsz       : 640
  batch       : 8
  device      : 0
  project     : c:\Users\Yasindu\Desktop\Stuff\5.ML Projects\deep-learning-projects\01_aerovision_lk\runs\baseline
  name        : yolov8s_640
  exist_ok    : True
  verbose     : True


In [5]:
# ── Train with MLflow tracking ────────────────────────────────────────────────
with mlflow.start_run(run_name='yolov8s-baseline-640') as run:
    mlflow.log_params({
        'model':     'yolov8s',
        'epochs':    TRAIN_PARAMS['epochs'],
        'imgsz':     TRAIN_PARAMS['imgsz'],
        'batch':     TRAIN_PARAMS['batch'],
        'device':    TRAIN_PARAMS['device'],
        'dataset':   'VisDrone2019-DET-9class',
        'optimizer': 'SGD',
    })

    model = YOLO('yolov8s.pt')
    results = model.train(**TRAIN_PARAMS)

    # ── Extract and log final metrics ─────────────────────────────────────────
    rd = results.results_dict
    box_loss_final = rd.get('train/box_loss',      None)
    map50          = rd.get('metrics/mAP50(B)',     None)
    map5095        = rd.get('metrics/mAP50-95(B)',  None)

    if box_loss_final is not None:
        mlflow.log_metric('train/box_loss_final', box_loss_final)
    if map50 is not None:
        mlflow.log_metric('val/mAP50',    map50)
    if map5095 is not None:
        mlflow.log_metric('val/mAP50-95', map5095)

    MLFLOW_RUN_ID = run.info.run_id

print()
print(f'Training complete. Run ID: {MLFLOW_RUN_ID}')
print(f'Best weights : {RESULTS_DIR / "yolov8s_640" / "weights" / "best.pt"}')
print(f'box_loss     : {box_loss_final}')
print(f'mAP@0.5      : {map50}')
print(f'mAP@0.5:0.95 : {map5095}')

New https://pypi.org/project/ultralytics/8.4.24 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.21  Python-3.11.15 torch-2.5.1 CUDA:0 (NVIDIA GeForce RTX 3050 6GB Laptop GPU, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\Users\Yasindu\Desktop\Stuff\5.ML Projects\deep-learning-projects\01_aerovision_lk\data\VisDrone_Dataset\visdrone_abs.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det

---
## 4. Evaluation on Val Set

In [6]:
best_weights = RESULTS_DIR / 'yolov8s_640' / 'weights' / 'best.pt'
assert best_weights.exists(), f'best.pt not found at {best_weights}. Did training complete?'

eval_model  = YOLO(best_weights)
val_results = eval_model.val(
    data=str(DATASET_YAML_ABS),
    imgsz=640,
    batch=8,
    verbose=True,
)

# ── Extract overall metrics ───────────────────────────────────────────────────
overall_map50   = float(val_results.results_dict.get('metrics/mAP50(B)',     0))
overall_map5095 = float(val_results.results_dict.get('metrics/mAP50-95(B)', 0))
overall_p       = float(val_results.results_dict.get('metrics/precision(B)', 0))
overall_r       = float(val_results.results_dict.get('metrics/recall(B)',    0))

# ── Extract per-class AP ──────────────────────────────────────────────────────
per_class_rows = []
if hasattr(val_results, 'box') and hasattr(val_results.box, 'ap50'):
    ap50_per_class   = val_results.box.ap50
    ap5095_per_class = val_results.box.ap
    class_indices    = val_results.box.ap_class_index
    for idx, cid in enumerate(class_indices):
        per_class_rows.append({
            'class_id':    int(cid),
            'class_name':  CLASS_NAMES[int(cid)] if int(cid) < NC else f'class_{cid}',
            'AP@0.5':      round(float(ap50_per_class[idx]),   4),
            'AP@0.5:0.95': round(float(ap5095_per_class[idx]), 4),
        })

per_class_df = pd.DataFrame(per_class_rows).sort_values('class_id').reset_index(drop=True)

print('Val set results — YOLOv8s baseline @ 640px')
print('=' * 55)
print(f'  mAP@0.5        : {overall_map50   * 100:.2f}%')
print(f'  mAP@0.5:0.95   : {overall_map5095 * 100:.2f}%')
print(f'  Precision      : {overall_p       * 100:.2f}%')
print(f'  Recall         : {overall_r       * 100:.2f}%')
print()
print('  Per-class AP@0.5:')
print(f'  {"Class":<15} {"AP@0.5":>8} {"AP@0.5:0.95":>12}')
print('  ' + '-' * 38)
for _, row in per_class_df.iterrows():
    print(f'  {row["class_name"]:<15} {row["AP@0.5"]*100:>7.2f}% {row["AP@0.5:0.95"]*100:>11.2f}%')
print('=' * 55)

Ultralytics 8.4.21  Python-3.11.15 torch-2.5.1 CUDA:0 (NVIDIA GeForce RTX 3050 6GB Laptop GPU, 6144MiB)
Model summary (fused): 73 layers, 11,129,067 parameters, 0 gradients, 28.5 GFLOPs
val: Fast image access  (ping: 0.10.1 ms, read: 557.4260.3 MB/s, size: 126.7 KB)
val: Scanning C:\Users\Yasindu\Desktop\Stuff\5.ML Projects\deep-learning-projects\01_aerovision_lk\data\VisDrone_Dataset\VisDrone2019-DET-val\labels.cache... 548 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 548/548  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 69/69 5.3it/s 13.1s0.2s
                   all        548      38759      0.557      0.409      0.441      0.261
            pedestrian        520       8844       0.55      0.408      0.445        0.2
                people        482       5125      0.596      0.286      0.352       0.13
               bicycle        364       1287      0.285      0.148      0.132     0.0558
                 

---
## 5. Save Metrics to CSV

In [7]:
# ── Build summary row ─────────────────────────────────────────────────────────
summary_row = {
    'model':         'yolov8s',
    'imgsz':         640,
    'epochs':        50,
    'batch':         8,
    'sahi':          False,
    'mAP50':         round(overall_map50,   4),
    'mAP50_95':      round(overall_map5095, 4),
    'precision':     round(overall_p,       4),
    'recall':        round(overall_r,       4),
    'weights':       str(WEIGHTS_DIR / 'yolov8s_baseline.pt'),
    'timestamp':     datetime.datetime.now().isoformat(timespec='seconds'),
}

# Append per-class APs as flat columns
for _, row in per_class_df.iterrows():
    summary_row[f'AP50_{row["class_name"]}']   = row['AP@0.5']
    summary_row[f'AP5095_{row["class_name"]}'] = row['AP@0.5:0.95']

csv_path = ANALYSIS_DIR / 'baseline_metrics.csv'

# Append to existing CSV or create new
new_row_df = pd.DataFrame([summary_row])
if csv_path.exists():
    existing = pd.read_csv(csv_path)
    combined = pd.concat([existing, new_row_df], ignore_index=True)
else:
    combined = new_row_df
combined.to_csv(csv_path, index=False)

print(f'Saved → {csv_path}')
print()
print(new_row_df[['model', 'imgsz', 'epochs', 'mAP50', 'mAP50_95']].to_string(index=False))

Saved → c:\Users\Yasindu\Desktop\Stuff\5.ML Projects\deep-learning-projects\01_aerovision_lk\analysis\baseline_metrics.csv

  model  imgsz  epochs  mAP50  mAP50_95
yolov8s    640      50 0.4408    0.2609


---
## 6. Save Best Weights

In [9]:
dest = WEIGHTS_DIR / 'yolov8s_baseline.pt'
shutil.copy(best_weights, dest)

size_mb = dest.stat().st_size / 1024 / 1024
print(f'Copied  : {best_weights}')
print(f'     -> : {dest}')
print(f'Size    : {size_mb:.1f} MB')

# Log artifact directly via client (no active-run conflict)
from mlflow.tracking import MlflowClient
mlflow.end_run()
client = MlflowClient()
client.log_artifact(MLFLOW_RUN_ID, str(dest),     artifact_path='weights')
client.log_artifact(MLFLOW_RUN_ID, str(csv_path), artifact_path='metrics')
print(f'Logged artifact to MLflow run {MLFLOW_RUN_ID}')


Copied  : c:\Users\Yasindu\Desktop\Stuff\5.ML Projects\deep-learning-projects\01_aerovision_lk\runs\baseline\yolov8s_640\weights\best.pt
     -> : c:\Users\Yasindu\Desktop\Stuff\5.ML Projects\deep-learning-projects\01_aerovision_lk\weights\yolov8s_baseline.pt
Size    : 21.5 MB
Logged artifact to MLflow run cae3cedd9e5a45758f23d5fd0af8b5a5


---
## 7. Failure Mode Visualization

The 5 most crowded val images — ground truth vs predictions.  
Missed GT boxes (IoU < 0.5 against all predictions) are shown as **red dashed** overlays.

In [10]:
# ── Helpers ───────────────────────────────────────────────────────────────────

def load_label(label_path: Path):
    """Return (boxes_norm [N,4], class_ids [N]) from a YOLO label file."""
    rows, cids = [], []
    for line in label_path.read_text().splitlines():
        parts = line.strip().split()
        if len(parts) == 5:
            cids.append(int(parts[0]))
            rows.append([float(x) for x in parts[1:]])
    boxes = np.array(rows, dtype=np.float32) if rows else np.zeros((0, 4), dtype=np.float32)
    return boxes, cids


def yolo_to_xyxy(boxes_norm, img_w, img_h):
    """Convert YOLO normalized [cx, cy, w, h] -> pixel [x1, y1, x2, y2]."""
    if len(boxes_norm) == 0:
        return np.zeros((0, 4), dtype=np.float32)
    cx, cy, bw, bh = boxes_norm[:, 0], boxes_norm[:, 1], boxes_norm[:, 2], boxes_norm[:, 3]
    x1 = (cx - bw / 2) * img_w
    y1 = (cy - bh / 2) * img_h
    x2 = (cx + bw / 2) * img_w
    y2 = (cy + bh / 2) * img_h
    return np.stack([x1, y1, x2, y2], axis=1).astype(np.float32)


def iou_matrix(gt_xyxy, pred_xyxy):
    """
    Vectorized IoU between every GT box and every predicted box.
    Returns matrix of shape [N_gt, N_pred].
    No external dependencies.
    """
    if len(gt_xyxy) == 0 or len(pred_xyxy) == 0:
        return np.zeros((len(gt_xyxy), len(pred_xyxy)), dtype=np.float32)

    # Expand dims for broadcasting: gt [N,1,4], pred [1,M,4]
    gt   = gt_xyxy[:, None, :]    # [N, 1, 4]
    pred = pred_xyxy[None, :, :]  # [1, M, 4]

    inter_x1 = np.maximum(gt[..., 0], pred[..., 0])
    inter_y1 = np.maximum(gt[..., 1], pred[..., 1])
    inter_x2 = np.minimum(gt[..., 2], pred[..., 2])
    inter_y2 = np.minimum(gt[..., 3], pred[..., 3])

    inter_w = np.maximum(0.0, inter_x2 - inter_x1)
    inter_h = np.maximum(0.0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h

    area_gt   = (gt_xyxy[:, 2]   - gt_xyxy[:, 0])   * (gt_xyxy[:, 3]   - gt_xyxy[:, 1])
    area_pred = (pred_xyxy[:, 2] - pred_xyxy[:, 0]) * (pred_xyxy[:, 3] - pred_xyxy[:, 1])

    union_area = area_gt[:, None] + area_pred[None, :] - inter_area
    union_area = np.maximum(union_area, 1e-6)

    return inter_area / union_area  # [N_gt, N_pred]


def get_most_crowded(labels_dir: Path, n=5):
    """Return top-n label files by annotation count."""
    files  = list(labels_dir.glob('*.txt'))
    counts = [(f, sum(1 for l in f.read_text().splitlines() if l.strip())) for f in files]
    return sorted(counts, key=lambda x: x[1], reverse=True)[:n]


# ── Find 5 most crowded val images ────────────────────────────────────────────
val_labels_dir = DATASET_YAML.parent / ds_cfg.get('val', 'VisDrone2019-DET-val/images')
val_labels_dir = val_labels_dir.parent / 'labels'  # swap images -> labels
val_images_dir = val_labels_dir.parent / 'images'

crowded = get_most_crowded(val_labels_dir, n=5)
print('5 most crowded val images selected for failure analysis:')
for lf, cnt in crowded:
    print(f'  {lf.name}  →  {cnt} annotations')

5 most crowded val images selected for failure analysis:
  0000295_02400_d_0000033.txt  →  317 annotations
  0000295_02000_d_0000031.txt  →  311 annotations
  0000295_01800_d_0000030.txt  →  306 annotations
  0000295_01000_d_0000026.txt  →  251 annotations
  0000291_03201_d_0000884.txt  →  229 annotations


In [11]:
# ── Run inference on selected images ──────────────────────────────────────────
infer_model = YOLO(WEIGHTS_DIR / 'yolov8s_baseline.pt')

IOU_THRESHOLD = 0.5  # GT box is 'missed' if max IoU with any prediction < this

fig, axes = plt.subplots(5, 2, figsize=(22, 28))
fig.suptitle(
    'Baseline YOLOv8s @ 640px — Failure Mode Analysis\n'
    '(Left: Ground Truth | Right: Predictions — red dashed = missed objects)',
    fontsize=13, fontweight='bold'
)

total_gt = total_pred = total_missed = 0

for row_idx, (label_path, ann_count) in enumerate(crowded):
    img_path = val_images_dir / (label_path.stem + '.jpg')
    img_bgr  = cv2.imread(str(img_path))
    img_h, img_w = img_bgr.shape[:2]

    # ── Ground truth ──────────────────────────────────────────────────────────
    gt_boxes_norm, gt_cids = load_label(label_path)
    gt_xyxy = yolo_to_xyxy(gt_boxes_norm, img_w, img_h)

    gt_canvas = img_bgr.copy()
    for (x1, y1, x2, y2), cid in zip(gt_xyxy.astype(int), gt_cids):
        color = COLORS_BGR[cid % NC]
        cv2.rectangle(gt_canvas, (x1, y1), (x2, y2), color, 1)

    # ── Predictions ───────────────────────────────────────────────────────────
    pred_result = infer_model.predict(
        source=str(img_path),
        imgsz=640,
        conf=0.25,
        iou=0.45,
        verbose=False,
    )[0]

    pred_xyxy  = pred_result.boxes.xyxy.cpu().numpy()   if len(pred_result.boxes) else np.zeros((0, 4))
    pred_confs = pred_result.boxes.conf.cpu().numpy()   if len(pred_result.boxes) else np.array([])
    pred_cids  = pred_result.boxes.cls.cpu().numpy().astype(int) if len(pred_result.boxes) else np.array([], dtype=int)

    # ── Identify missed GT boxes via vectorized IoU ───────────────────────────
    iou_mat   = iou_matrix(gt_xyxy, pred_xyxy)          # [N_gt, N_pred]
    max_iou   = iou_mat.max(axis=1) if iou_mat.size > 0 else np.zeros(len(gt_xyxy))
    missed_mask = max_iou < IOU_THRESHOLD

    n_missed = int(missed_mask.sum())
    total_gt     += len(gt_xyxy)
    total_pred   += len(pred_xyxy)
    total_missed += n_missed

    # ── Draw prediction canvas ────────────────────────────────────────────────
    pred_canvas = img_bgr.copy()

    # Draw predictions (solid, class-colored)
    for (x1, y1, x2, y2), cid, conf in zip(pred_xyxy.astype(int), pred_cids, pred_confs):
        color = COLORS_BGR[int(cid) % NC]
        cv2.rectangle(pred_canvas, (x1, y1), (x2, y2), color, 1)

    # Draw missed GT boxes (red dashed overlay)
    for (x1, y1, x2, y2) in gt_xyxy[missed_mask].astype(int):
        # Simulate dashed line with short segments
        pts = [
            ((x1, y1), (x2, y1)),
            ((x2, y1), (x2, y2)),
            ((x2, y2), (x1, y2)),
            ((x1, y2), (x1, y1)),
        ]
        for (p1, p2) in pts:
            dx = p2[0] - p1[0]; dy = p2[1] - p1[1]
            length = max(abs(dx), abs(dy), 1)
            dash = 6
            for i in range(0, length, dash * 2):
                t0 = i / length
                t1 = min((i + dash) / length, 1.0)
                sx = int(p1[0] + dx * t0); sy = int(p1[1] + dy * t0)
                ex = int(p1[0] + dx * t1); ey = int(p1[1] + dy * t1)
                cv2.line(pred_canvas, (sx, sy), (ex, ey), (0, 0, 255), 1)

    # ── Plot ──────────────────────────────────────────────────────────────────
    ax_gt   = axes[row_idx, 0]
    ax_pred = axes[row_idx, 1]

    ax_gt.imshow(cv2.cvtColor(gt_canvas, cv2.COLOR_BGR2RGB))
    ax_gt.set_title(f'GT: {len(gt_xyxy)} objects | {label_path.stem}', fontsize=9)
    ax_gt.axis('off')

    ax_pred.imshow(cv2.cvtColor(pred_canvas, cv2.COLOR_BGR2RGB))
    recall_img = (len(gt_xyxy) - n_missed) / max(len(gt_xyxy), 1)
    ax_pred.set_title(
        f'Pred: {len(pred_xyxy)} boxes | Missed: {n_missed}/{len(gt_xyxy)} '
        f'(recall {recall_img*100:.0f}%)',
        fontsize=9,
        color='darkred' if n_missed > len(gt_xyxy) * 0.4 else 'black'
    )
    ax_pred.axis('off')

# ── Legend ────────────────────────────────────────────────────────────────────
class_patches = [mpatches.Patch(color=[c/255 for c in col], label=name)
                 for name, col in zip(CLASS_NAMES, COLORS_RGB)]
missed_patch  = mpatches.Patch(facecolor='none', edgecolor='red',
                                linestyle='--', label='missed GT (IoU < 0.5)')
fig.legend(
    handles=class_patches + [missed_patch],
    loc='lower center', ncol=5,
    fontsize=8.5, frameon=True,
    bbox_to_anchor=(0.5, -0.01)
)

# ── Overall stats ─────────────────────────────────────────────────────────────
overall_recall = (total_gt - total_missed) / max(total_gt, 1)
fig.text(
    0.5, 0.995,
    f'5-image summary  |  GT: {total_gt}  Pred: {total_pred}  '
    f'Missed: {total_missed}  Recall: {overall_recall*100:.1f}%',
    ha='center', va='top', fontsize=10, color='#c0392b'
)

plt.tight_layout()
out_path = FIGURES_DIR / 'baseline_failures.png'
plt.savefig(out_path, dpi=120, bbox_inches='tight')
print(f'Saved → {out_path}')
plt.show()

print()
print(f'5-image failure summary:')
print(f'  Total GT boxes  : {total_gt}')
print(f'  Total predicted : {total_pred}')
print(f'  Total missed    : {total_missed}')
print(f'  Overall recall  : {overall_recall*100:.1f}%')

Saved → c:\Users\Yasindu\Desktop\Stuff\5.ML Projects\deep-learning-projects\01_aerovision_lk\reports\figures\baseline_failures.png


<Figure size 2200x2800 with 10 Axes>


5-image failure summary:
  Total GT boxes  : 1414
  Total predicted : 696
  Total missed    : 848
  Overall recall  : 40.0%


---
## 8. Summary

In [12]:
print('=' * 55)
print('  Baseline Results — YOLOv8s @ 640px input')
print('=' * 55)
print(f'  mAP@0.5        : {overall_map50   * 100:.2f}%  (target ~28%)')
print(f'  mAP@0.5:0.95   : {overall_map5095 * 100:.2f}%  (target ~16%)')
print(f'  Precision      : {overall_p       * 100:.2f}%')
print(f'  Recall         : {overall_r       * 100:.2f}%')
print()
print('  Per-class AP@0.5:')
for _, row in per_class_df.iterrows():
    bar = '#' * int(30 * row['AP@0.5'])
    print(f'    {row["class_name"]:<15} {row["AP@0.5"]*100:>5.1f}%  {bar}')
print()
print('  Context:')
print('  78.7% of VisDrone objects are < 50x50 px at native resolution.')
print('  At 640px input they shrink to ~33px — below reliable detection threshold.')
print()
print('  This is the baseline to beat with SAHI.')
print('  Target with SAHI: mAP@0.5 ~43% (+15pp)')
print()
print(f'  Weights saved  : {WEIGHTS_DIR / "yolov8s_baseline.pt"}')
print(f'  Metrics saved  : {ANALYSIS_DIR / "baseline_metrics.csv"}')
print(f'  Failure figure : {FIGURES_DIR / "baseline_failures.png"}')
print(f'  MLflow run ID  : {MLFLOW_RUN_ID}')
print('=' * 55)
print()
print('Next: research/03_sahi_inference.ipynb')

  Baseline Results — YOLOv8s @ 640px input
  mAP@0.5        : 44.08%  (target ~28%)
  mAP@0.5:0.95   : 26.09%  (target ~16%)
  Precision      : 55.73%
  Recall         : 40.94%

  Per-class AP@0.5:
    pedestrian       44.5%  #############
    people           35.2%  ##########
    bicycle          13.2%  ###
    car              80.3%  ########################
    van              44.5%  #############
    truck            37.5%  ###########
    three_wheeler    36.8%  ###########
    bus              59.5%  #################
    motor            45.2%  #############

  Context:
  78.7% of VisDrone objects are < 50x50 px at native resolution.
  At 640px input they shrink to ~33px — below reliable detection threshold.

  This is the baseline to beat with SAHI.
  Target with SAHI: mAP@0.5 ~43% (+15pp)

  Weights saved  : c:\Users\Yasindu\Desktop\Stuff\5.ML Projects\deep-learning-projects\01_aerovision_lk\weights\yolov8s_baseline.pt
  Metrics saved  : c:\Users\Yasindu\Desktop\Stuff\5.ML P

In [13]:
# ── Manual MLflow log — run this after training if the MLflow warning fired ───
import pandas as pd
from mlflow.tracking import MlflowClient

# Read final epoch metrics directly from results.csv
results_csv = RESULTS_DIR / 'yolov8s_640' / 'results.csv'
res = pd.read_csv(results_csv)
res.columns = res.columns.str.strip()
final = res.iloc[-1]

box_loss_final = float(final['train/box_loss'])
map50          = float(final['metrics/mAP50(B)'])
map5095        = float(final['metrics/mAP50-95(B)'])

print('Final epoch metrics from results.csv:')
print(f'  box_loss     : {box_loss_final:.5f}')
print(f'  mAP@0.5      : {map50*100:.2f}%')
print(f'  mAP@0.5:0.95 : {map5095*100:.2f}%')
print()

# Start a fresh run (avoids param-conflict error from the previous attempt)
mlflow.set_tracking_uri((PROJECT_ROOT / 'mlruns').as_uri())
mlflow.set_experiment('aerovision-lk')

with mlflow.start_run(run_name='yolov8s-baseline-640-fixed') as run:
    mlflow.log_params({
        'model':     'yolov8s',
        'epochs':    50,
        'imgsz':     640,
        'batch':     4,
        'device':    0,
        'dataset':   'VisDrone2019-DET-9class',
        'optimizer': 'AdamW',
    })
    mlflow.log_metric('train/box_loss_final', box_loss_final)
    mlflow.log_metric('val/mAP50',            map50)
    mlflow.log_metric('val/mAP50-95',         map5095)

    # Log full epoch curve as steps
    for _, row in res.iterrows():
        step = int(row.get('epoch', 0))
        mlflow.log_metric('train/box_loss', float(row['train/box_loss']),      step=step)
        mlflow.log_metric('val/mAP50',      float(row['metrics/mAP50(B)']),    step=step)
        mlflow.log_metric('val/mAP50-95',   float(row['metrics/mAP50-95(B)']), step=step)

    MLFLOW_RUN_ID = run.info.run_id

print(f'Logged to MLflow run: {MLFLOW_RUN_ID}')

Final epoch metrics from results.csv:
  box_loss     : 1.26408
  mAP@0.5      : 43.87%
  mAP@0.5:0.95 : 26.00%

Logged to MLflow run: 2959ca6a226e44eb80bf31302eecbe6b
